# Hospital Readmission Prediction - Deployment

## Objective
Deploy the trained model as a production-ready Streamlit dashboard for real-time predictions.

---

In [ ]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

# Add parent directory to path for imports
sys.path.append('..')

print("=" * 70)
print("DEPLOYMENT PREPARATION")
print("=" * 70)

## 1. Load Model and Related Objects

In [ ]:
# Load saved model and objects
model = joblib.load('../models/best_model.pkl')
feature_cols = joblib.load('../models/feature_cols.pkl')
model_metrics = joblib.load('../models/model_metrics.pkl')

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nModel Metrics:")
for key, value in model_metrics.items():
    print(f"  {key}: {value}")

## 2. Test Prediction Function

In [ ]:
# Create a function to make predictions with new patient data
def predict_readmission(patient_data, model, feature_cols):
    """
    Predict 30-day readmission risk for a patient.
    
    Parameters:
    -----------
    patient_data : dict
        Dictionary containing patient features
    model : trained model
        Trained classifier
    feature_cols : list
        List of feature column names
    
    Returns:
    --------
    dict : Prediction results
    """
    # Create DataFrame with all features
    df = pd.DataFrame(columns=feature_cols)
    df = df.append(patient_data, ignore_index=True)
    df = df.fillna(0)
    
    # Make prediction
    prediction = model.predict(df)[0]
    probability = model.predict_proba(df)[0][1]
    
    return {
        'prediction': int(prediction),
        'probability': float(probability),
        'risk_level': 'High' if probability > 0.5 else 'Medium' if probability > 0.3 else 'Low'
    }

# Test with sample patient data
sample_patient = {
    'time_in_hospital': 7,
    'num_lab_procedures': 45,
    'num_medications': 20,
    'num_procedures': 2,
    'number_inpatient': 2,
    'number_outpatient': 1,
    'number_emergency': 0,
    'num_diagnoses': 5,
    'precode': 1
}

result = predict_readmission(sample_patient, model, feature_cols)
print("Sample Prediction:")
print(f"  Patient Data: {sample_patient}")
print(f"  Prediction: {result['prediction']}")
print(f"  Probability: {result['probability']:.2%}")
print(f"  Risk Level: {result['risk_level']}")

## 3. Create Sample Data for Dashboard

In [ ]:
# Create sample patient data for testing
sample_patients = pd.DataFrame({
    'patient_id': range(1, 11),
    'time_in_hospital': [3, 7, 14, 2, 10, 5, 8, 1, 12, 6],
    'num_lab_procedures': [30, 45, 60, 20, 55, 35, 50, 15, 70, 40],
    'num_medications': [12, 20, 30, 8, 25, 15, 22, 6, 35, 18],
    'num_procedures': [1, 2, 4, 0, 3, 1, 2, 0, 5, 2],
    'number_inpatient': [0, 2, 5, 0, 3, 1, 2, 0, 4, 1],
    'number_outpatient': [2, 1, 0, 5, 1, 3, 2, 8, 0, 2],
    'number_emergency': [0, 1, 2, 0, 1, 0, 1, 0, 3, 0],
    'num_diagnoses': [3, 5, 8, 2, 6, 4, 5, 1, 9, 4],
    'precode': [0, 1, 1, 0, 1, 0, 1, 0, 1, 0]
})

# Make predictions for all sample patients
predictions = []
for idx, row in sample_patients.iterrows():
    patient_data = row.drop('patient_id').to_dict()
    result = predict_readmission(patient_data, model, feature_cols)
    predictions.append(result)

sample_patients['prediction'] = [p['prediction'] for p in predictions]
sample_patients['readmission_probability'] = [p['probability'] for p in predictions]
sample_patients['risk_level'] = [p['risk_level'] for p in predictions]

print("Sample Patient Predictions:")
print(sample_patients.to_string(index=False))

## 4. Verify Streamlit App

In [ ]:
# Check if app.py exists
app_path = '../app.py'
if os.path.exists(app_path):
    print(f"Streamlit app found at: {app_path}")
    
    # Read first few lines to verify structure
    with open(app_path, 'r') as f:
        lines = f.readlines()[:30]
    print("\nApp structure preview:")
    for line in lines:
        print(line.rstrip())
else:
    print(f"Streamlit app not found at: {app_path}")

## 5. Deployment Instructions

In [ ]:
print("=" * 70)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 70)
print("""
OPTION 1: RUN LOCALLY
------------------------
1. Install dependencies:
   pip install -r requirements.txt

2. Run the Streamlit app:
   streamlit run app.py

3. Open browser at: http://localhost:8501


OPTION 2: DEPLOY TO STREAMLIT CLOUD
------------------------------------
1. Push code to GitHub
2. Go to https://share.streamlit.io
3. Connect your GitHub repository
4. Select the repository and branch
5. Set main file path: app.py
6. Click Deploy!


OPTION 3: DEPLOY TO HUGGING FACE SPACES
----------------------------------------
1. Push code to GitHub
2. Go to https://huggingface.co/spaces
3. Create new Space (Streamlit)
4. Link your GitHub repository
5. Deploy!


REQUIRED FILES FOR DEPLOYMENT:
-------------------------------
  - app.py (Streamlit dashboard)
  - models/best_model.pkl (trained model)
  - models/feature_cols.pkl (feature names)
  - models/model_metrics.pkl (metrics)
  - requirements.txt (dependencies)
  - README.md (documentation)
""")

## 6. Final Summary

In [ ]:
print("=" * 70)
print("PROJECT COMPLETION SUMMARY")
print("=" * 70)
print(f"""
HOSPITAL READMISSION PREDICTION PROJECT - COMPLETE!

DELIVERABLES:
-------------
1. Data Exploration (EDA)
   - Dataset: 100k+ diabetic patient records
   - Key findings: Age, prior visits, hospital stay duration are key risk factors

2. Data Preprocessing
   - Handled missing values
   - Created binary target (30-day readmission)
   - Applied SMOTE for class imbalance
   - Train/Val/Test split: 70/15/15

3. Model Training
   - Models tested: Logistic Regression, Random Forest, XGBoost, LightGBM
   - Best model: {model_metrics['model_name']}
   - Test ROC-AUC: {model_metrics['test_roc_auc']:.4f}
   - Test Recall: {model_metrics['test_recall']:.4f}

4. Deployment
   - Streamlit dashboard for real-time predictions
   - Production-ready model saved as best_model.pkl

BUSINESS IMPACT:
-----------------
- Identify high-risk patients early
- Enable proactive intervention strategies
- Reduce 30-day readmission rates
- Save 10-20% in hospital readmission costs

NEXT STEPS:
-----------
- Deploy to production (Streamlit Cloud/Hugging Face)
- Set up model monitoring
- Integrate with hospital EHR systems
- Collect feedback for model improvement
""")